# Global Automotive Investment Database — Block 3

## US SEC point-in-time fundamentals module

This version loads the persisted Block 2 security master from its manifest and Parquet outputs. It does not execute Blocks 1 or 2 and does not depend on another notebook's live runtime.

### Responsibilities

- derive the SEC-eligible universe from the global security master;
- resolve SEC CIKs using official SEC mappings;
- retrieve submissions and Company Facts data with persistent caching;
- preserve filing and acceptance timestamps for point-in-time research;
- standardise selected XBRL concepts without discarding raw facts;
- link facts back to provisional global `security_id` and `issuer_id` values;
- persist Block 3 outputs for the global fundamentals layer and later modules.

Create a Colab secret named `SEC_USER_AGENT` before running. Use the format: 'Name email@address'.


In [ ]:
# 1. INSTALL / IMPORT DEPENDENCIES
# ------------------------------------------------

!pip -q install pandas numpy requests tqdm pyarrow

from __future__ import annotations
import json, os, re, time

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Mapping, Optional

import numpy as np
import pandas as pd
import requests

from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)

In [ ]:
# 2. USER SETTINGS, DIRECTORIES AND BLOCK 2 INPUT
# ------------------------------------------------

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy")
    try:
        # SEC automated-access identification.
        SEC_USER_AGENT = userdata.get("SEC_USER_AGENT").strip()
    except Exception as exc:
        raise ValueError(
            "Create a Colab secret named 'SEC_USER_AGENT' and enable notebook access."
        ) from exc
else:
    PROJECT_ROOT = Path("/content/global_automotive_investment_database")
    SEC_USER_AGENT = os.environ.get("SEC_USER_AGENT", "").strip()

if not SEC_USER_AGENT:
    raise ValueError("SEC_USER_AGENT is empty.")

DATA_ROOT = PROJECT_ROOT / "data"
BLOCK_2_OUTPUT_DIR = DATA_ROOT / "interim" / "block_2"
BLOCK_2_MANIFEST_PATH = BLOCK_2_OUTPUT_DIR / "block_2_manifest.json"
BLOCK_3_OUTPUT_DIR = DATA_ROOT / "interim" / "block_3"
BLOCK_3_MANIFEST_PATH = BLOCK_3_OUTPUT_DIR / "block_3_manifest.json"

# Persistent SEC cache: subsequent reruns reuse responses already downloaded.
CACHE_DIR = DATA_ROOT / "raw" / "sec_fundamentals_cache"

for directory in [BLOCK_3_OUTPUT_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Filing and fact settings.
ALLOWED_FORMS = {
    "10-K", "10-K/A", "10-Q", "10-Q/A",
    "20-F", "20-F/A", "40-F", "40-F/A",
    "6-K", "6-K/A",
}

# Point-in-time cutoff. Use None for all currently available information.
AS_OF_DATE = None  # example: "2025-12-31"

REQUEST_INTERVAL_SECONDS = 0.12
REQUEST_TIMEOUT_SECONDS = 60
MAX_RETRIES = 5
DOWNLOAD_COMPANYFACTS = True
DOWNLOAD_SUBMISSIONS = True
PERSIST_BLOCK_3_OUTPUTS = True
OVERWRITE_PERSISTED_OUTPUTS = True

# Optional development limit. Leave as None for the complete mapped SEC universe.
MAX_SEC_ENTITIES = None


def load_manifest_tables(
    manifest_path: Path,
    required_table_names: set[str],
) -> tuple[dict[str, pd.DataFrame], dict]:
    """Load required Parquet tables declared in an upstream manifest."""

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Required Block 2 manifest not found: {manifest_path}\n"
            "Run revised Block 2 fully before running Block 3."
        )

    with manifest_path.open("r", encoding="utf-8") as file:
        manifest = json.load(file)

    records = {
        item["table_name"]: item
        for item in manifest.get("tables", [])
    }

    missing = set(required_table_names).difference(records)

    if missing:
        raise RuntimeError(
            "Block 2 manifest is missing required tables: "
            f"{sorted(missing)}"
        )

    loaded = {}

    for table_name in sorted(required_table_names):
        table_path = Path(records[table_name]["path"])

        if not table_path.exists():
            raise FileNotFoundError(
                f"Manifest entry exists but Parquet file is missing: {table_path}"
            )

        loaded[table_name] = pd.read_parquet(table_path)

    return loaded, manifest


REQUIRED_BLOCK_2_TABLES = {
    "security_master_df",
    "issuer_master_df",
    "source_security_bridge_df",
    "security_identifier_history_df",
}

block_2_inputs, block_2_manifest = load_manifest_tables(
    BLOCK_2_MANIFEST_PATH,
    REQUIRED_BLOCK_2_TABLES,
)

security_master_df = block_2_inputs["security_master_df"]
issuer_master_df = block_2_inputs["issuer_master_df"]
source_security_bridge_df = block_2_inputs["source_security_bridge_df"]
security_identifier_history_df = block_2_inputs["security_identifier_history_df"]

print("Loaded persisted Block 2 inputs without executing earlier notebooks:")
for name, dataframe in block_2_inputs.items():
    print(f"  {name}: {len(dataframe):,} rows × {len(dataframe.columns):,} columns")

print("\nPersistent SEC cache:", CACHE_DIR)
print("Block 3 output directory:", BLOCK_3_OUTPUT_DIR)

invalid_user_agent = (
    "@" not in SEC_USER_AGENT
    or "example.com" in SEC_USER_AGENT.lower()
)

if invalid_user_agent:
    raise ValueError(
        "SEC_USER_AGENT must contain your real name and email address."
    )

Mounted at /content/drive
Loaded persisted Block 2 inputs without executing earlier notebooks:
  issuer_master_df: 566 rows × 16 columns
  security_identifier_history_df: 22,368 rows × 11 columns
  security_master_df: 512 rows × 34 columns
  source_security_bridge_df: 7,968 rows × 36 columns

Persistent SEC cache: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/raw/sec_fundamentals_cache
Block 3 output directory: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_3


In [ ]:
# 3. SEC SESSION, CACHE AND REQUEST HELPERS
# ------------------------------------------------

def validate_sec_user_agent(user_agent: str) -> None:
    if not isinstance(user_agent, str) or "@" not in user_agent or "example.com" in user_agent.lower():
        raise ValueError(
            "Update SEC_USER_AGENT with your real name/organisation and email before running Block 3."
        )


@dataclass
class SecClient:
    user_agent: str
    cache_dir: Path
    interval_seconds: float = 0.12
    timeout_seconds: int = 60
    max_retries: int = 5

    def __post_init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": self.user_agent,
            "Accept-Encoding": "gzip, deflate",
            "Host": "data.sec.gov",
        })
        self._last_request_at = 0.0

    def _cache_path(self, url: str) -> Path:
        safe = re.sub(r"[^A-Za-z0-9._-]+", "_", url.replace("https://", ""))
        return self.cache_dir / f"{safe}.json"

    def get_json(self, url: str, use_cache: bool = True) -> dict | list:
        path = self._cache_path(url)
        if use_cache and path.exists():
            with path.open("r", encoding="utf-8") as f:
                return json.load(f)

        for attempt in range(1, self.max_retries + 1):
            elapsed = time.monotonic() - self._last_request_at
            if elapsed < self.interval_seconds:
                time.sleep(self.interval_seconds - elapsed)

            try:
                response = self.session.get(url, timeout=self.timeout_seconds)
                self._last_request_at = time.monotonic()

                if response.status_code == 429:
                    time.sleep(min(2 ** attempt, 30))
                    continue

                response.raise_for_status()
                payload = response.json()
                with path.open("w", encoding="utf-8") as f:
                    json.dump(payload, f)
                return payload

            except (requests.RequestException, ValueError) as exc:
                if attempt == self.max_retries:
                    raise RuntimeError(f"SEC request failed after {attempt} attempts: {url}") from exc
                time.sleep(min(2 ** attempt, 30))

        raise RuntimeError(f"Unexpected SEC request failure: {url}")


validate_sec_user_agent(SEC_USER_AGENT)
sec_client = SecClient(
    user_agent=SEC_USER_AGENT,
    cache_dir=CACHE_DIR,
    interval_seconds=REQUEST_INTERVAL_SECONDS,
    timeout_seconds=REQUEST_TIMEOUT_SECONDS,
    max_retries=MAX_RETRIES,
)

In [ ]:
# 4. BUILD THE SEC UNIVERSE AND IDENTIFIER CANDIDATES FROM BLOCK 2
# ------------------------------------------------

def clean_ticker(value) -> Optional[str]:
    if pd.isna(value):
        return None

    value = str(value).strip().upper()
    value = re.sub(r"\s+", "", value)

    return value or None


def clean_cik(value) -> Optional[str]:
    if pd.isna(value):
        return None

    digits = re.sub(
        r"\D",
        "",
        str(value),
    )

    return digits.zfill(10) if digits else None


def first_existing_column(
    dataframe: pd.DataFrame,
    candidates: Iterable[str],
) -> Optional[str]:
    return next(
        (
            column
            for column in candidates
            if column in dataframe.columns
        ),
        None,
    )


def coalesce_existing_columns(
    dataframe: pd.DataFrame,
    candidates: Iterable[str],
    default=pd.NA,
) -> pd.Series:
    available = [
        column
        for column in candidates
        if column in dataframe.columns
    ]

    if not available:
        return pd.Series(
            default,
            index=dataframe.index,
            dtype="object",
        )

    output = dataframe[
        available[0]
    ].copy()

    for column in available[1:]:
        output = output.combine_first(
            dataframe[column]
        )

    return output


def normalise_security_master(
    master: pd.DataFrame,
) -> pd.DataFrame:
    if not isinstance(
        master,
        pd.DataFrame,
    ) or master.empty:
        raise ValueError(
            "Persisted security_master_df is missing or empty."
        )

    output = pd.DataFrame(
        index=master.index
    )

    output["security_id"] = (
        coalesce_existing_columns(
            master,
            [
                "security_id",
                "canonical_security_id",
            ],
        )
    )

    output["issuer_id"] = (
        coalesce_existing_columns(
            master,
            [
                "issuer_id",
                "canonical_issuer_id",
            ],
        )
    )

    output["issuer_name"] = (
        coalesce_existing_columns(
            master,
            [
                "issuer_name",
                "canonical_issuer_name",
                "security_name",
                "name",
            ],
        )
    )

    output["country"] = (
        coalesce_existing_columns(
            master,
            [
                "country",
                "issuer_country",
                "domicile_country",
            ],
        )
    )

    output["ticker"] = (
        coalesce_existing_columns(
            master,
            [
                "ticker",
                "primary_ticker",
                "ticker_normalised",
                "source_ticker",
            ],
        )
        .map(clean_ticker)
    )

    output["exchange"] = (
        coalesce_existing_columns(
            master,
            [
                "exchange",
                "exchange_code",
                "mic",
                "primary_exchange",
            ],
        )
    )

    output["isin"] = (
        coalesce_existing_columns(
            master,
            [
                "isin",
                "ISIN",
            ],
        )
    )

    output["cik"] = (
        coalesce_existing_columns(
            master,
            [
                "cik",
                "sec_cik",
                "cik_normalised",
            ],
        )
        .map(clean_cik)
    )

    output["is_primary_security"] = (
        coalesce_existing_columns(
            master,
            [
                "is_primary_security",
                "is_primary_listing",
                "primary_security_flag",
            ],
            default=False,
        )
        .fillna(False)
        .astype("boolean")
    )

    return (
        output
        .dropna(
            subset=[
                "security_id",
            ]
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )


def extract_identifier_candidates(
    dataframe: pd.DataFrame,
    source_table: str,
) -> pd.DataFrame:
    if (
        not isinstance(
            dataframe,
            pd.DataFrame,
        )
        or dataframe.empty
    ):
        return pd.DataFrame(
            columns=[
                "security_id",
                "issuer_id",
                "ticker",
                "cik",
                "identifier_type",
                "identifier_value",
                "effective_start",
                "effective_end",
                "source_table",
            ]
        )

    output = pd.DataFrame(
        index=dataframe.index
    )

    output["security_id"] = (
        coalesce_existing_columns(
            dataframe,
            [
                "security_id",
                "canonical_security_id",
                "target_security_id",
            ],
        )
    )

    output["issuer_id"] = (
        coalesce_existing_columns(
            dataframe,
            [
                "issuer_id",
                "canonical_issuer_id",
                "target_issuer_id",
            ],
        )
    )

    output["ticker"] = (
        coalesce_existing_columns(
            dataframe,
            [
                "ticker",
                "source_ticker",
                "identifier_value",
                "symbol",
            ],
        )
        .map(clean_ticker)
    )

    output["identifier_type"] = (
        coalesce_existing_columns(
            dataframe,
            [
                "identifier_type",
                "id_type",
                "source_identifier_type",
            ],
        )
        .astype("string")
        .str.upper()
    )

    output["identifier_value"] = (
        coalesce_existing_columns(
            dataframe,
            [
                "identifier_value",
                "source_identifier",
                "identifier",
                "id_value",
            ],
        )
    )

    direct_cik = (
        coalesce_existing_columns(
            dataframe,
            [
                "cik",
                "sec_cik",
                "cik_normalised",
            ],
        )
        .map(clean_cik)
    )

    cik_from_identifier = (
        output[
            "identifier_value"
        ]
        .where(
            output[
                "identifier_type"
            ]
            .isin(
                {
                    "CIK",
                    "SEC_CIK",
                }
            )
        )
        .map(clean_cik)
    )

    output["cik"] = (
        direct_cik.combine_first(
            cik_from_identifier
        )
    )

    output["effective_start"] = (
        pd.to_datetime(
            coalesce_existing_columns(
                dataframe,
                [
                    "effective_start",
                    "valid_from",
                    "start_date",
                ],
            ),
            errors="coerce",
        )
    )

    output["effective_end"] = (
        pd.to_datetime(
            coalesce_existing_columns(
                dataframe,
                [
                    "effective_end",
                    "valid_to",
                    "end_date",
                ],
            ),
            errors="coerce",
        )
    )

    output["source_table"] = (
        source_table
    )

    return (
        output[
            output["cik"].notna()
            | output["ticker"].notna()
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )


security_master_normalised_df = (
    normalise_security_master(
        security_master_df
    )
)

identifier_candidates_df = pd.concat(
    [
        extract_identifier_candidates(
            source_security_bridge_df,
            "source_security_bridge_df",
        ),
        extract_identifier_candidates(
            security_identifier_history_df,
            "security_identifier_history_df",
        ),
    ],
    ignore_index=True,
    sort=False,
)

sec_universe_seed_df = (
    security_master_normalised_df
    .merge(
        identifier_candidates_df[
            [
                "security_id",
                "cik",
                "ticker",
                "source_table",
            ]
        ]
        .rename(
            columns={
                "cik": "cik_candidate",
                "ticker": "ticker_candidate",
                "source_table": (
                    "identifier_source_table"
                ),
            }
        ),
        on="security_id",
        how="left",
        validate="1:m",
    )
)

sec_universe_seed_df["cik"] = (
    sec_universe_seed_df["cik"]
    .combine_first(
        sec_universe_seed_df[
            "cik_candidate"
        ]
    )
    .map(clean_cik)
)

sec_universe_seed_df["ticker"] = (
    sec_universe_seed_df["ticker"]
    .combine_first(
        sec_universe_seed_df[
            "ticker_candidate"
        ]
    )
    .map(clean_ticker)
)

sec_universe_seed_df = (
    sec_universe_seed_df[
        sec_universe_seed_df[
            "ticker"
        ].notna()
        | sec_universe_seed_df[
            "cik"
        ].notna()
    ]
    .drop_duplicates(
        [
            "security_id",
            "issuer_id",
            "ticker",
            "cik",
        ]
    )
    .reset_index(drop=True)
)

if MAX_SEC_ENTITIES is not None:
    sec_universe_seed_df = (
        sec_universe_seed_df
        .head(
            int(
                MAX_SEC_ENTITIES
            )
        )
        .copy()
    )

print(
    "SEC universe seed rows:",
    f"{len(sec_universe_seed_df):,}",
)

print(
    "Seed rows with issuer_id:",
    int(
        sec_universe_seed_df[
            "issuer_id"
        ].notna().sum()
    ),
)

print(
    "Seed rows with direct or historical CIK:",
    int(
        sec_universe_seed_df[
            "cik"
        ].notna().sum()
    ),
)

display(
    sec_universe_seed_df.head(20)
)

/tmp/ipykernel_2540/1445974772.py:186: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


SEC universe seed rows: 825
Seed rows with issuer_id: 825
Seed rows with direct or historical CIK: 0


,security_id,issuer_id,issuer_name,country,ticker,exchange,isin,cik,is_primary_security,cik_candidate,ticker_candidate,identifier_source_table
0,GAS_000EFF8BBFFA00B37D9E,GAI_D8F28DDEBA5977F9D69A,"Xiamen Tungsten Co., Ltd.",CN,CNE000001D15,None,CNE000001D15,None,False,None,CNE000001D15,security_identifier_history_df
1,GAS_000EFF8BBFFA00B37D9E,GAI_D8F28DDEBA5977F9D69A,"Xiamen Tungsten Co., Ltd.",CN,300300SEC2FOC4PL5N49,None,CNE000001D15,None,False,None,300300SEC2FOC4PL5N49,security_identifier_history_df
2,GAS_012FF9156F0DFDA56EDC,GAI_B4458F7BDC67FFDB71E4,TI Fluid Systems PLC,GB,000000000,None,GB00BYQB9V88,None,False,None,000000000,security_identifier_history_df
3,GAS_012FF9156F0DFDA56EDC,GAI_B4458F7BDC67FFDB71E4,TI Fluid Systems PLC,GB,GB00BYQB9V88,None,GB00BYQB9V88,None,False,None,GB00BYQB9V88,security_identifier_history_df
4,GAS_012FF9156F0DFDA56EDC,GAI_B4458F7BDC67FFDB71E4,TI Fluid Systems PLC,GB,5493001T9RXVD6OAWY46,None,GB00BYQB9V88,None,False,None,5493001T9RXVD6OAWY46,security_identifier_history_df
5,GAS_018F78AA0262BAA2EF41,GAI_CAE0EE7F2C53BE3D2D56,Merdeka Copper Gold Tbk PT,ID,MDKA,None,ID1000134406,None,False,NaN,MDKA,source_security_bridge_df
6,GAS_01F546D04C0C4F72A262,GAI_5CD5347816CF9C46E768,Aston Martin Lagonda Global Ho,GB,AML,None,GB00BFXZC448,None,False,NaN,AML,source_security_bridge_df
7,GAS_021CBE637C40EA92408A,GAI_2F165C93319FA20296BA,MaxLinear Inc,US,MXL,None,US57776J1007,None,False,NaN,MXL,source_security_bridge_df
8,GAS_0222D68E994B13D9EF6C,GAI_766E3386DD1DD1EF6C87,Cirrus Logic Inc,US,CRUS,None,US1727551004,None,False,NaN,CRUS,source_security_bridge_df
9,GAS_0257192FE3F982588E78,GAI_691EB6978F5517E828B5,Nissha Co Ltd,JP,000000000,None,JP3713200008,None,False,None,000000000,security_identifier_history_df


In [ ]:
# 5. OFFICIAL SEC TICKER–CIK DIRECTORY AND AUTHORITATIVE CIK BRIDGES
# ------------------------------------------------

SEC_TICKER_URL = (
    "https://www.sec.gov/files/"
    "company_tickers_exchange.json"
)


def get_sec_ticker_directory() -> pd.DataFrame:
    cache_path = (
        CACHE_DIR
        / "company_tickers_exchange.json"
    )

    if cache_path.exists():
        payload = json.loads(
            cache_path.read_text(
                encoding="utf-8"
            )
        )

    else:
        headers = {
            "User-Agent": SEC_USER_AGENT,
            "Accept-Encoding": (
                "gzip, deflate"
            ),
        }

        response = requests.get(
            SEC_TICKER_URL,
            headers=headers,
            timeout=(
                REQUEST_TIMEOUT_SECONDS
            ),
        )

        response.raise_for_status()

        payload = response.json()

        cache_path.write_text(
            json.dumps(payload),
            encoding="utf-8",
        )

    if (
        isinstance(payload, dict)
        and "data" in payload
        and "fields" in payload
    ):
        directory = pd.DataFrame(
            payload["data"],
            columns=payload["fields"],
        )

    else:
        directory = (
            pd.DataFrame(payload)
            .T
            .reset_index(drop=True)
        )

    rename_map = {
        "cik": "cik",
        "cik_str": "cik",
        "name": "sec_company_name",
        "ticker": "sec_ticker",
        "exchange": "sec_exchange",
    }

    directory = directory.rename(
        columns={
            key: value
            for key, value in (
                rename_map.items()
            )
            if key in directory.columns
        }
    )

    directory["cik"] = (
        directory["cik"]
        .map(clean_cik)
    )

    directory["sec_ticker"] = (
        directory["sec_ticker"]
        .map(clean_ticker)
    )

    return (
        directory[
            [
                column
                for column in [
                    "cik",
                    "sec_company_name",
                    "sec_ticker",
                    "sec_exchange",
                ]
                if column
                in directory.columns
            ]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )


def map_universe_to_cik(
    seed: pd.DataFrame,
    directory: pd.DataFrame,
) -> pd.DataFrame:
    direct = seed.copy()

    direct["ticker"] = (
        direct["ticker"]
        .map(clean_ticker)
    )

    direct["cik"] = (
        direct["cik"]
        .map(clean_cik)
    )

    ticker_lookup = (
        directory
        .dropna(
            subset=[
                "sec_ticker",
                "cik",
            ]
        )
        .copy()
    )

    mapped = direct.merge(
        ticker_lookup,
        how="left",
        left_on="ticker",
        right_on="sec_ticker",
        suffixes=(
            "",
            "_directory",
        ),
    )

    mapped["resolved_cik"] = (
        mapped["cik"]
        .combine_first(
            mapped[
                "cik_directory"
            ]
        )
        .map(clean_cik)
    )

    mapped["cik_mapping_method"] = (
        np.select(
            [
                mapped["cik"].notna(),
                mapped[
                    "cik_directory"
                ].notna(),
            ],
            [
                "BLOCK_2_DIRECT_OR_HISTORICAL_CIK",
                "SEC_TICKER_DIRECTORY",
            ],
            default="UNRESOLVED",
        )
    )

    mapped["ticker_match_count"] = (
        mapped
        .groupby(
            [
                "security_id",
                "ticker",
            ],
            dropna=False,
        )[
            "cik_directory"
        ]
        .transform("nunique")
    )

    mapped[
        "cik_mapping_ambiguous"
    ] = (
        mapped[
            "ticker_match_count"
        ]
        .fillna(0)
        .gt(1)
    )

    mapped[
        "issuer_id_missing"
    ] = mapped[
        "issuer_id"
    ].isna()

    return mapped


sec_ticker_directory_df = (
    get_sec_ticker_directory()
)

sec_universe_map_df = (
    map_universe_to_cik(
        sec_universe_seed_df,
        sec_ticker_directory_df,
    )
)

sec_resolved_universe_df = (
    sec_universe_map_df
    .loc[
        sec_universe_map_df[
            "resolved_cik"
        ].notna()
        & ~sec_universe_map_df[
            "cik_mapping_ambiguous"
        ]
        & sec_universe_map_df[
            "issuer_id"
        ].notna()
    ]
    .drop_duplicates(
        subset=[
            "security_id",
            "resolved_cik",
        ]
    )
    .reset_index(drop=True)
)

sec_unresolved_universe_df = (
    sec_universe_map_df
    .loc[
        sec_universe_map_df[
            "resolved_cik"
        ].isna()
        | sec_universe_map_df[
            "cik_mapping_ambiguous"
        ]
        | sec_universe_map_df[
            "issuer_id"
        ].isna()
    ]
    .reset_index(drop=True)
)


# ------------------------------------------------
# CIK → SECURITY AND CIK → ISSUER BRIDGES
# ------------------------------------------------

sec_cik_security_bridge_df = (
    sec_resolved_universe_df[
        [
            "resolved_cik",
            "security_id",
            "issuer_id",
            "issuer_name",
            "ticker",
            "exchange",
            "isin",
            "country",
            "is_primary_security",
            "cik_mapping_method",
            "sec_company_name",
            "sec_exchange",
        ]
    ]
    .rename(
        columns={
            "resolved_cik": "cik",
        }
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

issuer_bridge_quality_df = (
    sec_cik_security_bridge_df
    .groupby(
        "cik",
        dropna=False,
    )
    .agg(
        issuer_id_count=(
            "issuer_id",
            "nunique",
        ),
        security_count=(
            "security_id",
            "nunique",
        ),
        ticker_count=(
            "ticker",
            "nunique",
        ),
    )
    .reset_index()
)

sec_cik_issuer_bridge_candidates_df = (
    sec_cik_security_bridge_df
    .merge(
        issuer_bridge_quality_df,
        on="cik",
        how="left",
        validate="m:1",
    )
)

sec_cik_issuer_bridge_candidates_df[
    "_primary_rank"
] = (
    ~sec_cik_issuer_bridge_candidates_df[
        "is_primary_security"
    ]
    .fillna(False)
    .astype(bool)
).astype(int)

sec_cik_issuer_bridge_df = (
    sec_cik_issuer_bridge_candidates_df[
        sec_cik_issuer_bridge_candidates_df[
            "issuer_id_count"
        ].eq(1)
    ]
    .sort_values(
        [
            "cik",
            "_primary_rank",
            "security_id",
        ],
        na_position="last",
    )
    .drop_duplicates(
        "cik",
        keep="first",
    )
    .rename(
        columns={
            "security_id": (
                "primary_security_id"
            ),
            "ticker": (
                "primary_ticker"
            ),
            "exchange": (
                "primary_exchange"
            ),
            "isin": (
                "primary_isin"
            ),
        }
    )
    [
        [
            "cik",
            "issuer_id",
            "issuer_name",
            "country",
            "primary_security_id",
            "primary_ticker",
            "primary_exchange",
            "primary_isin",
            "security_count",
            "cik_mapping_method",
            "sec_company_name",
        ]
    ]
    .reset_index(drop=True)
)

sec_cik_issuer_conflicts_df = (
    sec_cik_issuer_bridge_candidates_df[
        sec_cik_issuer_bridge_candidates_df[
            "issuer_id_count"
        ].ne(1)
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------
# STANDARD DOWNSTREAM ISSUER-CENTRIC CONTRACTS
# ------------------------------------------------

usa_economic_issuer_universe_df = (
    issuer_master_df.copy()
)

issuer_id_column = first_existing_column(
    usa_economic_issuer_universe_df,
    [
        "issuer_id",
        "canonical_issuer_id",
    ],
)

issuer_name_column = first_existing_column(
    usa_economic_issuer_universe_df,
    [
        "issuer_name",
        "canonical_issuer_name",
        "name",
    ],
)

if issuer_id_column is None:
    raise KeyError(
        "issuer_master_df does not contain issuer_id."
    )

usa_economic_issuer_universe_df = (
    usa_economic_issuer_universe_df[
        usa_economic_issuer_universe_df[
            issuer_id_column
        ].isin(
            sec_cik_issuer_bridge_df[
                "issuer_id"
            ]
        )
    ]
    .copy()
)

if issuer_id_column != "issuer_id":
    usa_economic_issuer_universe_df = (
        usa_economic_issuer_universe_df
        .rename(
            columns={
                issuer_id_column: (
                    "issuer_id"
                )
            }
        )
    )

if (
    issuer_name_column is not None
    and issuer_name_column
    != "issuer_name"
):
    usa_economic_issuer_universe_df = (
        usa_economic_issuer_universe_df
        .rename(
            columns={
                issuer_name_column: (
                    "issuer_name"
                )
            }
        )
    )

usa_economic_issuer_universe_df = (
    usa_economic_issuer_universe_df
    .drop_duplicates(
        "issuer_id"
    )
    .reset_index(drop=True)
)

usa_issuer_security_universe_df = (
    sec_cik_security_bridge_df
    .rename(
        columns={
            "cik": "sec_cik",
        }
    )
    .copy()
)

usa_entity_relationship_graph_df = (
    pd.DataFrame(
        columns=[
            "from_entity_id",
            "to_entity_id",
            "relationship_type",
            "effective_start",
            "effective_end",
            "confidence",
            "source_system",
            "source_region",
        ]
    )
)

usa_preferred_accounting_source_df = (
    sec_cik_issuer_bridge_df[
        [
            "issuer_id",
            "cik",
        ]
    ]
    .rename(
        columns={
            "cik": "preferred_source_entity_id",
        }
    )
    .assign(
        preferred_source_system=(
            "SEC_EDGAR_COMPANYFACTS"
        ),
        preferred_source_region="USA",
        source_confidence=1.0,
        selection_basis=(
            "AUTHORITATIVE_SEC_REGISTRANT_CIK"
        ),
    )
)

print(
    "Resolved SEC security mappings:",
    f"{len(sec_resolved_universe_df):,}",
)

print(
    "Confirmed CIK-to-issuer mappings:",
    f"{len(sec_cik_issuer_bridge_df):,}",
)

print(
    "CIK-to-issuer conflicts:",
    f"{len(sec_cik_issuer_conflicts_df):,}",
)

display(
    sec_cik_issuer_bridge_df.head(20)
)

Resolved SEC security mappings: 93
Confirmed CIK-to-issuer mappings: 84
CIK-to-issuer conflicts: 6


,cik,issuer_id,issuer_name,country,primary_security_id,primary_ticker,primary_exchange,primary_isin,security_count,cik_mapping_method,sec_company_name
0,0000002488,GAI_34F1613C884F5729743E,Advanced Micro Devices Inc,US,GAS_7B77344EA4DB7EC2F450,AMD,None,US0079031078,1,SEC_TICKER_DIRECTORY,ADVANCED MICRO DEVICES INC
1,0000004127,GAI_45979D2A46D24D62955E,Skyworks Solutions Inc,US,GAS_303B28BF837EB57E3757,SWKS,None,US83088M1027,1,SEC_TICKER_DIRECTORY,"SKYWORKS SOLUTIONS, INC."
2,0000006281,GAI_1D758CFD82BFA008D269,Analog Devices Inc,US,GAS_A5C788E414083BE813B3,ADI,None,US0326541051,1,SEC_TICKER_DIRECTORY,ANALOG DEVICES INC
3,0000026172,GAI_C0CFC39BABE30ADE14FD,Cummins Inc,US,GAS_361D302F41890AFD9F4F,CMI,None,US2310211063,1,SEC_TICKER_DIRECTORY,CUMMINS INC
4,0000037996,GAI_A5FA100A985C44709B37,Ford Motor Co,US,GAS_F9EE243C18608349B1A8,F,None,US3453708600,1,SEC_TICKER_DIRECTORY,FORD MOTOR CO
5,0000050863,GAI_20EBE040D79E9FA58E6A,Intel Corp,US,GAS_DD5DB962F7D897B060EB,INTC,None,US4581401001,1,SEC_TICKER_DIRECTORY,INTEL CORP
6,0000065770,GAI_CA431A2E420DB71C2B37,MicroVision Inc,US,GAS_DF27BDC5CB902B421305,MVIS,None,US5949603048,1,SEC_TICKER_DIRECTORY,"MICROVISION, INC."
7,0000075362,GAI_F69FED6C415FD084BF7A,PACCAR Inc,US,GAS_486AEEAF5B5EF72EDF0F,PCAR,None,US6937181088,1,SEC_TICKER_DIRECTORY,PACCAR INC
8,0000097476,GAI_D27BAB5E21620F872FFF,Texas Instruments Inc,US,GAS_37F8C4B640C06F7FE4B3,TXN,None,US8825081040,1,SEC_TICKER_DIRECTORY,TEXAS INSTRUMENTS INC
9,0000101295,GAI_14CA0396C936391F94E9,Peugeot SA,FR,GAS_4335B112661A29610412,UG,None,FR0000121501,1,SEC_TICKER_DIRECTORY,UNITED GUARDIAN INC


In [ ]:
# 6. DOWNLOAD AND FLATTEN SEC SUBMISSIONS
# ------------------------------------------------

def submissions_url(cik: str) -> str:
    return f"https://data.sec.gov/submissions/CIK{clean_cik(cik)}.json"


def companyfacts_url(cik: str) -> str:
    return f"https://data.sec.gov/api/xbrl/companyfacts/CIK{clean_cik(cik)}.json"


def flatten_recent_submissions(payload: Mapping, cik: str) -> pd.DataFrame:
    recent = payload.get("filings", {}).get("recent", {})
    if not recent:
        return pd.DataFrame()

    frame = pd.DataFrame(recent)
    if frame.empty:
        return frame

    frame["cik"] = clean_cik(cik)
    frame["entity_name"] = payload.get("name")
    frame["sic"] = payload.get("sic")
    frame["sic_description"] = payload.get("sicDescription")
    frame["fiscal_year_end"] = payload.get("fiscalYearEnd")

    rename = {
        "accessionNumber": "accession_number",
        "filingDate": "filing_date",
        "reportDate": "report_date",
        "acceptanceDateTime": "acceptance_datetime",
        "primaryDocument": "primary_document",
        "primaryDocDescription": "primary_document_description",
        "filmNumber": "film_number",
        "isXBRL": "is_xbrl",
        "isInlineXBRL": "is_inline_xbrl",
    }
    frame = frame.rename(columns={k: v for k, v in rename.items() if k in frame.columns})
    return frame


submissions_frames = []
submission_download_log = []

if DOWNLOAD_SUBMISSIONS:
    for cik in tqdm(sec_resolved_universe_df["resolved_cik"].dropna().unique(), desc="SEC submissions"):
        try:
            payload = sec_client.get_json(submissions_url(cik))
            frame = flatten_recent_submissions(payload, cik)
            if not frame.empty:
                submissions_frames.append(frame)
            submission_download_log.append({"cik": cik, "status": "OK", "rows": len(frame), "error": None})
        except Exception as exc:
            submission_download_log.append({"cik": cik, "status": "ERROR", "rows": 0, "error": str(exc)})

sec_submissions_raw_df = pd.concat(submissions_frames, ignore_index=True) if submissions_frames else pd.DataFrame()
sec_submission_download_log_df = pd.DataFrame(submission_download_log)

if not sec_submissions_raw_df.empty:
    for col in ["filing_date", "report_date"]:
        if col in sec_submissions_raw_df.columns:
            sec_submissions_raw_df[col] = pd.to_datetime(sec_submissions_raw_df[col], errors="coerce")
    if "acceptance_datetime" in sec_submissions_raw_df.columns:
        sec_submissions_raw_df["acceptance_datetime"] = pd.to_datetime(
            sec_submissions_raw_df["acceptance_datetime"], errors="coerce", utc=True
        )

    sec_submissions_df = sec_submissions_raw_df.loc[
        sec_submissions_raw_df["form"].isin(ALLOWED_FORMS)
    ].copy()

    if AS_OF_DATE is not None:
        cutoff = pd.Timestamp(AS_OF_DATE, tz="UTC")
        sec_submissions_df = sec_submissions_df.loc[
            sec_submissions_df["acceptance_datetime"].fillna(
                sec_submissions_df["filing_date"].dt.tz_localize("UTC")
            ) <= cutoff
        ].copy()
else:
    sec_submissions_df = pd.DataFrame()

print(f"Relevant SEC filing rows: {len(sec_submissions_df):,}")

SEC submissions:   0%|          | 0/87 [00:00<?, ?it/s]

Relevant SEC filing rows: 9,898


In [ ]:
# 7. DOWNLOAD AND FLATTEN COMPANY FACTS
# ------------------------------------------------

def flatten_companyfacts(payload: Mapping, cik: str) -> pd.DataFrame:
    rows = []
    entity_name = payload.get("entityName")

    for taxonomy, taxonomy_facts in payload.get("facts", {}).items():
        for tag, fact in taxonomy_facts.items():
            label = fact.get("label")
            description = fact.get("description")
            for unit, observations in fact.get("units", {}).items():
                for obs in observations:
                    row = dict(obs)
                    row.update({
                        "cik": clean_cik(cik),
                        "entity_name": entity_name,
                        "taxonomy": taxonomy,
                        "tag": tag,
                        "label": label,
                        "description": description,
                        "unit": unit,
                    })
                    rows.append(row)

    return pd.DataFrame(rows)


companyfacts_frames = []
companyfacts_download_log = []

if DOWNLOAD_COMPANYFACTS:
    for cik in tqdm(sec_resolved_universe_df["resolved_cik"].dropna().unique(), desc="SEC Company Facts"):
        try:
            payload = sec_client.get_json(companyfacts_url(cik))
            frame = flatten_companyfacts(payload, cik)
            if not frame.empty:
                companyfacts_frames.append(frame)
            companyfacts_download_log.append({"cik": cik, "status": "OK", "rows": len(frame), "error": None})
        except Exception as exc:
            companyfacts_download_log.append({"cik": cik, "status": "ERROR", "rows": 0, "error": str(exc)})

sec_companyfacts_raw_df = pd.concat(companyfacts_frames, ignore_index=True) if companyfacts_frames else pd.DataFrame()
sec_companyfacts_download_log_df = pd.DataFrame(companyfacts_download_log)

if not sec_companyfacts_raw_df.empty:
    sec_companyfacts_raw_df = sec_companyfacts_raw_df.rename(columns={
        "accn": "accession_number",
        "fy": "fiscal_year",
        "fp": "fiscal_period",
        "filed": "filing_date",
        "start": "period_start",
        "end": "period_end",
        "val": "reported_value",
    })

    for col in ["filing_date", "period_start", "period_end"]:
        if col in sec_companyfacts_raw_df.columns:
            sec_companyfacts_raw_df[col] = pd.to_datetime(sec_companyfacts_raw_df[col], errors="coerce")

    sec_companyfacts_raw_df = sec_companyfacts_raw_df.loc[
        sec_companyfacts_raw_df["form"].isin(ALLOWED_FORMS)
    ].copy()

print(f"Raw XBRL fact observations: {len(sec_companyfacts_raw_df):,}")

SEC Company Facts:   0%|          | 0/87 [00:00<?, ?it/s]

Raw XBRL fact observations: 1,306,194


In [ ]:
# 8. ATTACH FILING-LEVEL POINT-IN-TIME METADATA
# ------------------------------------------------

def build_filing_metadata(submissions: pd.DataFrame) -> pd.DataFrame:
    if submissions.empty:
        return pd.DataFrame()

    wanted = [
        "cik", "accession_number", "form", "filing_date", "report_date",
        "acceptance_datetime", "primary_document", "is_xbrl", "is_inline_xbrl",
        "entity_name", "sic", "sic_description", "fiscal_year_end",
    ]
    wanted = [c for c in wanted if c in submissions.columns]
    out = submissions[wanted].drop_duplicates(subset=["cik", "accession_number"]).copy()

    if "acceptance_datetime" in out.columns:
        out["available_datetime"] = out["acceptance_datetime"]
    else:
        out["available_datetime"] = pd.to_datetime(out["filing_date"], utc=True)

    out["available_date"] = pd.to_datetime(out["available_datetime"], utc=True).dt.normalize()
    out["is_amendment"] = out["form"].astype(str).str.endswith("/A")
    return out


sec_filing_metadata_df = build_filing_metadata(sec_submissions_df)

if not sec_companyfacts_raw_df.empty:
    sec_facts_pit_df = sec_companyfacts_raw_df.merge(
        sec_filing_metadata_df,
        on=["cik", "accession_number"],
        how="left",
        suffixes=("_fact", "_filing"),
        validate="many_to_one",
    )

    # Company Facts provides filed date even when a submission join is unavailable.
    fallback_available = pd.to_datetime(sec_facts_pit_df.get("filing_date_fact"), errors="coerce", utc=True)
    sec_facts_pit_df["available_datetime"] = sec_facts_pit_df["available_datetime"].combine_first(fallback_available)
    sec_facts_pit_df["available_date"] = pd.to_datetime(sec_facts_pit_df["available_datetime"], utc=True).dt.normalize()

    if AS_OF_DATE is not None:
        cutoff = pd.Timestamp(AS_OF_DATE, tz="UTC")
        sec_facts_pit_df = sec_facts_pit_df.loc[sec_facts_pit_df["available_datetime"] <= cutoff].copy()
else:
    sec_facts_pit_df = pd.DataFrame()

print(f"Point-in-time fact rows: {len(sec_facts_pit_df):,}")

Point-in-time fact rows: 1,306,194


In [ ]:
# 9. EXPANDED CANONICAL SEC CONCEPT DICTIONARY
# ------------------------------------------------

# Canonical names match Block 4 V4. Regional source tags differ, but downstream
# models receive the same standard_concept field across the US and Europe.

SEC_CANONICAL_MAPPINGS = [
    # standard_concept, taxonomy, tag, priority, statement_type,
    # expected_period_type, expected_unit_family, core_tier

    # ------------------------------------------------
    # INCOME STATEMENT
    # ------------------------------------------------
    ("revenue", "us-gaap", "RevenueFromContractWithCustomerExcludingAssessedTax", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("revenue", "us-gaap", "SalesRevenueNet", 2, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("revenue", "us-gaap", "Revenues", 3, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("revenue", "ifrs-full", "Revenue", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),

    ("cost_of_revenue", "us-gaap", "CostOfRevenue", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("cost_of_revenue", "us-gaap", "CostOfGoodsAndServicesSold", 2, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("cost_of_revenue", "ifrs-full", "CostOfSales", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),

    ("gross_profit", "us-gaap", "GrossProfit", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("gross_profit", "ifrs-full", "GrossProfit", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),

    ("operating_income", "us-gaap", "OperatingIncomeLoss", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("operating_income", "ifrs-full", "ProfitLossFromOperatingActivities", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),

    ("profit_before_tax", "us-gaap", "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("profit_before_tax", "us-gaap", "IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeLossFromEquityMethodInvestments", 2, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("profit_before_tax", "ifrs-full", "ProfitLossBeforeTax", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),

    ("income_tax_expense", "us-gaap", "IncomeTaxExpenseBenefit", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("income_tax_expense", "ifrs-full", "IncomeTaxExpenseContinuingOperations", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),

    ("net_income", "us-gaap", "NetIncomeLoss", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("net_income", "us-gaap", "ProfitLoss", 2, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("net_income", "ifrs-full", "ProfitLoss", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),

    ("net_income_attributable_to_owners", "us-gaap", "NetIncomeLossAvailableToCommonStockholdersBasic", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("net_income_attributable_to_owners", "us-gaap", "NetIncomeLossAttributableToParent", 2, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),
    ("net_income_attributable_to_owners", "ifrs-full", "ProfitLossAttributableToOwnersOfParent", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 1),

    ("net_income_attributable_to_nci", "us-gaap", "NetIncomeLossAttributableToNoncontrollingInterest", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("net_income_attributable_to_nci", "ifrs-full", "ProfitLossAttributableToNoncontrollingInterests", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),

    ("basic_eps", "us-gaap", "EarningsPerShareBasic", 1, "INCOME_STATEMENT", "DURATION", "PER_SHARE", 1),
    ("basic_eps", "ifrs-full", "BasicEarningsLossPerShare", 1, "INCOME_STATEMENT", "DURATION", "PER_SHARE", 1),
    ("diluted_eps", "us-gaap", "EarningsPerShareDiluted", 1, "INCOME_STATEMENT", "DURATION", "PER_SHARE", 1),
    ("diluted_eps", "ifrs-full", "DilutedEarningsLossPerShare", 1, "INCOME_STATEMENT", "DURATION", "PER_SHARE", 1),

    ("basic_weighted_average_shares", "us-gaap", "WeightedAverageNumberOfSharesOutstandingBasic", 1, "INCOME_STATEMENT", "DURATION", "SHARES", 1),
    ("diluted_weighted_average_shares", "us-gaap", "WeightedAverageNumberOfDilutedSharesOutstanding", 1, "INCOME_STATEMENT", "DURATION", "SHARES", 1),

    ("research_and_development_expense", "us-gaap", "ResearchAndDevelopmentExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("research_and_development_expense", "ifrs-full", "ResearchAndDevelopmentExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("selling_expense", "us-gaap", "SellingExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("general_and_administrative_expense", "us-gaap", "GeneralAndAdministrativeExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("selling_general_and_administrative_expense", "us-gaap", "SellingGeneralAndAdministrativeExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("employee_benefit_expense", "us-gaap", "LaborAndRelatedExpense", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),

    ("depreciation_expense", "us-gaap", "Depreciation", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("amortisation_expense", "us-gaap", "AmortizationOfIntangibleAssets", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("depreciation_and_amortisation", "us-gaap", "DepreciationDepletionAndAmortization", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("impairment_loss", "us-gaap", "ImpairmentOfLongLivedAssetsHeldForUse", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("restructuring_expense", "us-gaap", "RestructuringCharges", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 3),

    ("finance_income", "ifrs-full", "FinanceIncome", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("finance_costs", "ifrs-full", "FinanceCosts", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("interest_expense", "us-gaap", "InterestExpenseNonOperating", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("interest_expense", "us-gaap", "InterestExpense", 2, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("interest_income", "us-gaap", "InterestIncomeExpenseNonoperatingNet", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),

    ("share_of_profit_equity_method", "us-gaap", "IncomeLossFromEquityMethodInvestments", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),
    ("share_of_profit_equity_method", "ifrs-full", "ShareOfProfitLossOfAssociatesAndJointVenturesAccountedForUsingEquityMethod", 1, "INCOME_STATEMENT", "DURATION", "MONETARY", 2),

    # ------------------------------------------------
    # COMPREHENSIVE INCOME
    # ------------------------------------------------
    ("other_comprehensive_income", "us-gaap", "OtherComprehensiveIncomeLossNetOfTax", 1, "COMPREHENSIVE_INCOME", "DURATION", "MONETARY", 2),
    ("other_comprehensive_income", "ifrs-full", "OtherComprehensiveIncome", 1, "COMPREHENSIVE_INCOME", "DURATION", "MONETARY", 2),
    ("comprehensive_income", "us-gaap", "ComprehensiveIncomeNetOfTax", 1, "COMPREHENSIVE_INCOME", "DURATION", "MONETARY", 2),
    ("comprehensive_income", "ifrs-full", "ComprehensiveIncome", 1, "COMPREHENSIVE_INCOME", "DURATION", "MONETARY", 2),
    ("comprehensive_income_attributable_to_owners", "us-gaap", "ComprehensiveIncomeNetOfTaxAttributableToParent", 1, "COMPREHENSIVE_INCOME", "DURATION", "MONETARY", 2),

    # ------------------------------------------------
    # ASSETS
    # ------------------------------------------------
    ("total_assets", "us-gaap", "Assets", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("total_assets", "ifrs-full", "Assets", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("current_assets", "us-gaap", "AssetsCurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("current_assets", "ifrs-full", "CurrentAssets", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("noncurrent_assets", "us-gaap", "AssetsNoncurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("noncurrent_assets", "ifrs-full", "NoncurrentAssets", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),

    ("cash_and_cash_equivalents", "us-gaap", "CashAndCashEquivalentsAtCarryingValue", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("cash_and_cash_equivalents", "us-gaap", "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents", 2, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("cash_and_cash_equivalents", "ifrs-full", "CashAndCashEquivalents", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("restricted_cash", "us-gaap", "RestrictedCashAndCashEquivalentsCurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),
    ("short_term_investments", "us-gaap", "ShortTermInvestments", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),

    ("trade_receivables", "us-gaap", "AccountsReceivableNetCurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("trade_receivables", "ifrs-full", "TradeReceivables", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("other_receivables", "us-gaap", "OtherReceivablesNetCurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("finance_receivables", "us-gaap", "FinanceReceivablesNet", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),

    ("inventory", "us-gaap", "InventoryNet", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("inventory", "ifrs-full", "Inventories", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("raw_material_inventory", "us-gaap", "InventoryRawMaterialsAndSuppliesNetOfAllowancesCustomerAdvancesAndProgressBillings", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),
    ("work_in_progress_inventory", "us-gaap", "InventoryWorkInProcessNetOfAllowancesCustomerAdvancesAndProgressBillings", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),
    ("finished_goods_inventory", "us-gaap", "InventoryFinishedGoodsNetOfAllowancesCustomerAdvancesAndProgressBillings", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),

    ("property_plant_equipment", "us-gaap", "PropertyPlantAndEquipmentNet", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("property_plant_equipment", "ifrs-full", "PropertyPlantAndEquipment", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("right_of_use_assets", "us-gaap", "OperatingLeaseRightOfUseAsset", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("goodwill", "us-gaap", "Goodwill", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("goodwill", "ifrs-full", "Goodwill", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("intangible_assets", "us-gaap", "FiniteLivedIntangibleAssetsNet", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("intangible_assets", "us-gaap", "IndefiniteLivedIntangibleAssetsExcludingGoodwill", 2, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("intangible_assets", "ifrs-full", "IntangibleAssetsOtherThanGoodwill", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("capitalised_development_costs", "ifrs-full", "DevelopmentCosts", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),
    ("investments_in_associates", "us-gaap", "EquityMethodInvestments", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("deferred_tax_assets", "us-gaap", "DeferredTaxAssetsNetCurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("deferred_tax_assets", "us-gaap", "DeferredTaxAssetsNetNoncurrent", 2, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("pension_assets", "us-gaap", "DefinedBenefitPlanAssetsForPlanBenefits", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),

    # ------------------------------------------------
    # LIABILITIES AND EQUITY
    # ------------------------------------------------
    ("total_liabilities", "us-gaap", "Liabilities", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("total_liabilities", "ifrs-full", "Liabilities", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("current_liabilities", "us-gaap", "LiabilitiesCurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("current_liabilities", "ifrs-full", "CurrentLiabilities", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("noncurrent_liabilities", "us-gaap", "LiabilitiesNoncurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("noncurrent_liabilities", "ifrs-full", "NoncurrentLiabilities", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),

    ("trade_payables", "us-gaap", "AccountsPayableCurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("trade_payables", "ifrs-full", "TradePayables", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("other_payables", "us-gaap", "OtherAccountsPayableCurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("contract_liabilities", "us-gaap", "ContractWithCustomerLiabilityCurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("contract_liabilities", "us-gaap", "DeferredRevenueCurrent", 2, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),

    ("short_term_debt", "us-gaap", "ShortTermBorrowings", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("short_term_debt", "us-gaap", "LongTermDebtCurrent", 2, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("long_term_debt", "us-gaap", "LongTermDebtNoncurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("long_term_debt", "us-gaap", "LongTermDebt", 2, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("total_borrowings", "us-gaap", "LongTermDebtAndFinanceLeaseObligationsCurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("total_borrowings", "ifrs-full", "Borrowings", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),

    ("current_lease_liabilities", "us-gaap", "OperatingLeaseLiabilityCurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("noncurrent_lease_liabilities", "us-gaap", "OperatingLeaseLiabilityNoncurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),

    ("provisions_current", "ifrs-full", "CurrentProvisions", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("provisions_noncurrent", "ifrs-full", "NoncurrentProvisions", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("warranty_provisions", "us-gaap", "StandardProductWarrantyAccrual", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),
    ("restructuring_provisions", "us-gaap", "RestructuringReserve", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 3),
    ("pension_liabilities", "us-gaap", "DefinedBenefitPensionPlanLiabilitiesNoncurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("deferred_tax_liabilities", "us-gaap", "DeferredTaxLiabilitiesNoncurrent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),

    ("total_equity", "us-gaap", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("total_equity", "us-gaap", "StockholdersEquity", 2, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("total_equity", "ifrs-full", "Equity", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("equity_attributable_to_owners", "us-gaap", "StockholdersEquity", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("equity_attributable_to_owners", "ifrs-full", "EquityAttributableToOwnersOfParent", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 1),
    ("noncontrolling_interests", "us-gaap", "MinorityInterest", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("share_capital", "us-gaap", "CommonStocksIncludingAdditionalPaidInCapital", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("share_premium", "us-gaap", "AdditionalPaidInCapital", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("retained_earnings", "us-gaap", "RetainedEarningsAccumulatedDeficit", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("treasury_shares", "us-gaap", "TreasuryStockValue", 1, "BALANCE_SHEET", "INSTANT", "MONETARY", 2),
    ("shares_outstanding", "dei", "EntityCommonStockSharesOutstanding", 1, "BALANCE_SHEET", "INSTANT", "SHARES", 1),

    # ------------------------------------------------
    # CASH FLOW
    # ------------------------------------------------
    ("operating_cash_flow", "us-gaap", "NetCashProvidedByUsedInOperatingActivities", 1, "CASH_FLOW", "DURATION", "MONETARY", 1),
    ("operating_cash_flow", "ifrs-full", "CashFlowsFromUsedInOperatingActivities", 1, "CASH_FLOW", "DURATION", "MONETARY", 1),
    ("investing_cash_flow", "us-gaap", "NetCashProvidedByUsedInInvestingActivities", 1, "CASH_FLOW", "DURATION", "MONETARY", 1),
    ("financing_cash_flow", "us-gaap", "NetCashProvidedByUsedInFinancingActivities", 1, "CASH_FLOW", "DURATION", "MONETARY", 1),

    ("capital_expenditure", "us-gaap", "PaymentsToAcquirePropertyPlantAndEquipment", 1, "CASH_FLOW", "DURATION", "MONETARY", 1),
    ("capital_expenditure", "ifrs-full", "PurchaseOfPropertyPlantAndEquipment", 1, "CASH_FLOW", "DURATION", "MONETARY", 1),
    ("intangible_asset_purchases", "us-gaap", "PaymentsToAcquireProductiveAssets", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("business_acquisition_cash_outflow", "us-gaap", "PaymentsToAcquireBusinessesNetOfCashAcquired", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("business_disposal_cash_inflow", "us-gaap", "ProceedsFromDividendsReceived", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),

    ("debt_issuance", "us-gaap", "ProceedsFromIssuanceOfLongTermDebt", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("debt_repayment", "us-gaap", "RepaymentsOfLongTermDebt", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("lease_payments", "us-gaap", "FinanceLeasePrincipalPayments", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),

    ("dividends_paid", "us-gaap", "PaymentsOfDividends", 1, "CASH_FLOW", "DURATION", "MONETARY", 1),
    ("share_repurchases", "us-gaap", "PaymentsForRepurchaseOfCommonStock", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("share_issuance_proceeds", "us-gaap", "ProceedsFromStockOptionsExercised", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),

    ("interest_paid", "us-gaap", "InterestPaidNet", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("interest_received", "ifrs-full", "InterestReceived", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("income_taxes_paid", "us-gaap", "IncomeTaxesPaidNet", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),
    ("cash_change", "us-gaap", "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalentsPeriodIncreaseDecreaseIncludingExchangeRateEffect", 1, "CASH_FLOW", "DURATION", "MONETARY", 2),

    # ------------------------------------------------
    # OPTIONAL AUTOMOTIVE / OPERATING METRICS
    # ------------------------------------------------
    ("vehicle_sales_volume", "custom", "VehicleSalesVolume", 1, "OPERATING_METRIC", "DURATION", "COUNT", 3),
    ("vehicle_production_volume", "custom", "VehicleProductionVolume", 1, "OPERATING_METRIC", "DURATION", "COUNT", 3),
    ("automotive_revenue", "custom", "AutomotiveRevenue", 1, "SEGMENT", "DURATION", "MONETARY", 3),
    ("financial_services_revenue", "custom", "FinancialServicesRevenue", 1, "SEGMENT", "DURATION", "MONETARY", 3),
    ("automotive_debt", "custom", "AutomotiveDebt", 1, "SEGMENT", "INSTANT", "MONETARY", 3),
    ("financial_services_debt", "custom", "FinancialServicesDebt", 1, "SEGMENT", "INSTANT", "MONETARY", 3),
]


sec_standard_concept_dictionary_df = pd.DataFrame(
    SEC_CANONICAL_MAPPINGS,
    columns=[
        "standard_concept",
        "taxonomy",
        "tag",
        "priority",
        "statement_type",
        "expected_period_type",
        "expected_unit_family",
        "core_tier",
    ],
)

sec_standard_concept_dictionary_df["is_core"] = (
    sec_standard_concept_dictionary_df["core_tier"] <= 2
)

sec_standard_concept_dictionary_df["aggregation_policy"] = np.where(
    sec_standard_concept_dictionary_df["expected_period_type"].eq("INSTANT"),
    "LATEST_INSTANT",
    "PERIOD_VALUE",
)

print(
    "Canonical standard concepts:",
    sec_standard_concept_dictionary_df["standard_concept"].nunique(),
)

print(
    "SEC source-concept mapping rows:",
    len(sec_standard_concept_dictionary_df),
)


Canonical standard concepts: 100
SEC source-concept mapping rows: 147


In [ ]:
# 10. MAP FACTS TO THE CANONICAL SCHEMA AND RESOLVE DUPLICATES
# ------------------------------------------------

def classify_sec_period_type(row) -> str:
    start = pd.to_datetime(row.get("period_start"), errors="coerce")
    end = pd.to_datetime(row.get("period_end"), errors="coerce")

    if pd.isna(start) or start == end:
        return "INSTANT"

    return "DURATION"


def classify_sec_unit_family(unit_value) -> str:
    if pd.isna(unit_value):
        return "UNKNOWN"

    text = str(unit_value).lower()

    if "shares" in text and "/" not in text:
        return "SHARES"

    if (
        "shares" in text
        and (
            "/" in text
            or "per" in text
        )
    ):
        return "PER_SHARE"

    if any(
        token in text
        for token in [
            "usd", "eur", "gbp", "jpy", "cny", "chf",
            "sek", "nok", "dkk", "cad", "aud",
        ]
    ):
        return "MONETARY"

    if any(
        token in text
        for token in ["pure", "number", "count", "vehicle", "unit"]
    ):
        return "COUNT"

    if "%" in text or "percent" in text:
        return "PERCENTAGE"

    return "OTHER"


def standardise_sec_facts(
    facts: pd.DataFrame,
    dictionary: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    if facts.empty:
        return pd.DataFrame(), pd.DataFrame()

    working = facts.copy()

    working["reported_value"] = pd.to_numeric(
        working["reported_value"],
        errors="coerce",
    )

    working["period_type"] = working.apply(
        classify_sec_period_type,
        axis=1,
    )

    working["observed_unit_family"] = (
        working["unit"]
        .map(classify_sec_unit_family)
    )

    mapped = working.merge(
        dictionary,
        on=["taxonomy", "tag"],
        how="left",
        validate="many_to_many",
    )

    matched = mapped[
        mapped["standard_concept"].notna()
    ].copy()

    if matched.empty:
        return matched, matched.copy()

    matched["duration_days"] = (
        pd.to_datetime(
            matched["period_end"],
            errors="coerce",
        )
        - pd.to_datetime(
            matched["period_start"],
            errors="coerce",
        )
    ).dt.days

    matched["period_type_match"] = (
        matched["period_type"]
        == matched["expected_period_type"]
    )

    matched["unit_family_match"] = (
        matched["observed_unit_family"]
        == matched["expected_unit_family"]
    )

    matched["is_numeric_fact"] = (
        matched["reported_value"].notna()
    )

    matched["is_standard_taxonomy"] = (
        matched["taxonomy"].isin(
            ["us-gaap", "ifrs-full", "dei"]
        )
    )

    matched["selection_score"] = (
        matched["priority"] * 100
        + (~matched["period_type_match"]) * 20
        + (~matched["unit_family_match"]) * 10
        + (~matched["is_numeric_fact"]) * 5
        + (~matched["is_standard_taxonomy"]) * 2
    )

    key = [
        "cik",
        "standard_concept",
        "period_start",
        "period_end",
        "fiscal_year",
        "fiscal_period",
        "unit",
        "accession_number",
    ]

    key = [
        column
        for column in key
        if column in matched.columns
    ]

    matched = (
        matched
        .sort_values(
            key
            + [
                "selection_score",
                "available_datetime",
                "tag",
            ],
            ascending=(
                [True] * len(key)
                + [True, False, True]
            ),
            na_position="last",
        )
        .reset_index(drop=True)
    )

    matched["concept_selection_rank"] = (
        matched
        .groupby(
            key,
            dropna=False,
        )
        .cumcount()
        + 1
    )

    matched["is_selected_standard_fact"] = (
        matched["concept_selection_rank"].eq(1)
    )

    selected = matched[
        matched["is_selected_standard_fact"]
    ].copy()

    return matched, selected


sec_fundamentals_mapped_df, sec_fundamentals_standardised_df = (
    standardise_sec_facts(
        sec_facts_pit_df,
        sec_standard_concept_dictionary_df,
    )
)


# ------------------------------------------------
# UNMAPPED AND AUTOMOTIVE EXTENSION INVENTORIES
# ------------------------------------------------

if sec_facts_pit_df.empty:

    sec_unmapped_concept_inventory_df = pd.DataFrame()
    sec_automotive_extension_candidates_df = pd.DataFrame()

else:

    mapped_keys = (
        sec_standard_concept_dictionary_df[
            ["taxonomy", "tag"]
        ]
        .drop_duplicates()
    )

    raw_with_mapping_flag = sec_facts_pit_df.merge(
        mapped_keys.assign(is_mapped=True),
        on=["taxonomy", "tag"],
        how="left",
    )

    unmapped = raw_with_mapping_flag[
        raw_with_mapping_flag["is_mapped"].isna()
    ].copy()

    sec_unmapped_concept_inventory_df = (
        unmapped
        .groupby(
            [
                "taxonomy",
                "tag",
                "label",
                "unit",
            ],
            dropna=False,
        )
        .agg(
            fact_rows=("reported_value", "size"),
            issuer_count=("cik", "nunique"),
            filing_count=("accession_number", "nunique"),
            numeric_fact_share=(
                "reported_value",
                lambda series: pd.to_numeric(
                    series,
                    errors="coerce",
                ).notna().mean(),
            ),
            earliest_available=(
                "available_datetime",
                "min",
            ),
            latest_available=(
                "available_datetime",
                "max",
            ),
        )
        .reset_index()
        .sort_values(
            ["issuer_count", "fact_rows"],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    automotive_pattern = re.compile(
        r"(vehicle|automotive|production|deliver|warranty|"
        r"dealer|finance.?receivable|battery|electric|"
        r"development.?cost|restructur|segment)",
        flags=re.IGNORECASE,
    )

    extension_text = (
        sec_unmapped_concept_inventory_df["tag"]
        .astype("string")
        + " "
        + sec_unmapped_concept_inventory_df["label"]
        .astype("string")
    )

    sec_automotive_extension_candidates_df = (
        sec_unmapped_concept_inventory_df[
            extension_text.str.contains(
                automotive_pattern,
                na=False,
            )
        ]
        .copy()
        .reset_index(drop=True)
    )


print(
    "Mapped candidate facts:",
    f"{len(sec_fundamentals_mapped_df):,}",
)

print(
    "Selected standardised facts:",
    f"{len(sec_fundamentals_standardised_df):,}",
)

print(
    "Unmapped source concepts inventoried:",
    f"{len(sec_unmapped_concept_inventory_df):,}",
)

print(
    "Automotive extension candidates:",
    f"{len(sec_automotive_extension_candidates_df):,}",
)


Mapped candidate facts: 402,604
Selected standardised facts: 387,948
Unmapped source concepts inventoried: 7,315
Automotive extension candidates: 175


/tmp/ipykernel_2540/1920711053.py:279: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  extension_text.str.contains(


In [ ]:
# 11. ATTACH AUTHORITATIVE ISSUER AND SECURITY IDENTIFIERS
# ------------------------------------------------

def attach_issuer_ids(
    dataframe: pd.DataFrame,
    issuer_bridge: pd.DataFrame,
) -> pd.DataFrame:
    if dataframe.empty:
        return dataframe.copy()

    if "cik" not in dataframe.columns:
        raise KeyError(
            "The SEC table does not contain cik."
        )

    bridge = (
        issuer_bridge[
            [
                "cik",
                "issuer_id",
                "issuer_name",
                "country",
                "primary_security_id",
                "primary_ticker",
                "primary_exchange",
                "primary_isin",
            ]
        ]
        .drop_duplicates(
            "cik"
        )
    )

    output = dataframe.merge(
        bridge,
        on="cik",
        how="left",
        validate="m:1",
        suffixes=(
            "",
            "_bridge",
        ),
    )

    if "issuer_name_bridge" in output.columns:
        if "issuer_name" not in output.columns:
            output["issuer_name"] = pd.NA

        output["issuer_name"] = (
            output[
                "issuer_name_bridge"
            ]
            .combine_first(
                output["issuer_name"]
            )
        )

    output["security_id"] = (
        output[
            "primary_security_id"
        ]
    )

    output["ticker"] = (
        output[
            "primary_ticker"
        ]
    )

    output["exchange"] = (
        output[
            "primary_exchange"
        ]
    )

    output["isin"] = (
        output[
            "primary_isin"
        ]
    )

    output[
        "issuer_link_status"
    ] = np.where(
        output["issuer_id"].notna(),
        "LINKED",
        "UNRESOLVED_CIK_TO_ISSUER",
    )

    return output


def attach_all_security_ids(
    dataframe: pd.DataFrame,
    security_bridge: pd.DataFrame,
) -> pd.DataFrame:
    if dataframe.empty:
        return dataframe.copy()

    bridge = (
        security_bridge[
            [
                "cik",
                "security_id",
                "issuer_id",
                "issuer_name",
                "ticker",
                "exchange",
                "isin",
                "country",
                "is_primary_security",
            ]
        ]
        .drop_duplicates()
    )

    return dataframe.merge(
        bridge,
        on=[
            "cik",
            "issuer_id",
        ],
        how="left",
        validate="m:m",
        suffixes=(
            "",
            "_security",
        ),
    )


# Preserve the CIK-level selected tables for audit.
sec_fundamentals_standardised_cik_df = (
    sec_fundamentals_standardised_df.copy()
)

sec_filing_metadata_cik_df = (
    sec_filing_metadata_df.copy()
)


# The authoritative standardised outputs now carry issuer_id.
sec_fundamentals_standardised_df = (
    attach_issuer_ids(
        sec_fundamentals_standardised_cik_df,
        sec_cik_issuer_bridge_df,
    )
)

sec_filing_metadata_df = (
    attach_issuer_ids(
        sec_filing_metadata_cik_df,
        sec_cik_issuer_bridge_df,
    )
)


# Security-expanded tables remain available separately.
sec_fundamentals_security_linked_df = (
    attach_all_security_ids(
        sec_fundamentals_standardised_df,
        sec_cik_security_bridge_df,
    )
)

sec_filing_metadata_linked_df = (
    attach_all_security_ids(
        sec_filing_metadata_df,
        sec_cik_security_bridge_df,
    )
)


# Explicit aliases used by Block 10.
usa_fundamentals_standardised_df = (
    sec_fundamentals_standardised_df
)

usa_filing_metadata_df = (
    sec_filing_metadata_df
)

usa_standard_concept_dictionary_df = (
    sec_standard_concept_dictionary_df
)


sec_issuer_link_quality_df = pd.DataFrame({
    "metric": [
        "standardised_fact_rows",
        "fact_rows_with_issuer_id",
        "fact_rows_missing_issuer_id",
        "fact_issuer_count",
        "fact_primary_security_count",
        "filing_rows",
        "filing_rows_with_issuer_id",
        "filing_rows_missing_issuer_id",
        "filing_issuer_count",
        "confirmed_cik_issuer_bridges",
        "conflicted_cik_issuer_bridges",
    ],
    "value": [
        len(
            sec_fundamentals_standardised_df
        ),
        int(
            sec_fundamentals_standardised_df[
                "issuer_id"
            ].notna().sum()
        ),
        int(
            sec_fundamentals_standardised_df[
                "issuer_id"
            ].isna().sum()
        ),
        sec_fundamentals_standardised_df[
            "issuer_id"
        ].nunique(),
        sec_fundamentals_standardised_df[
            "security_id"
        ].nunique(),
        len(
            sec_filing_metadata_df
        ),
        int(
            sec_filing_metadata_df[
                "issuer_id"
            ].notna().sum()
        ),
        int(
            sec_filing_metadata_df[
                "issuer_id"
            ].isna().sum()
        ),
        sec_filing_metadata_df[
            "issuer_id"
        ].nunique(),
        len(
            sec_cik_issuer_bridge_df
        ),
        len(
            sec_cik_issuer_conflicts_df
        ),
    ],
})

display(
    sec_issuer_link_quality_df
)

if (
    len(
        sec_fundamentals_standardised_df
    ) > 0
    and sec_fundamentals_standardised_df[
        "issuer_id"
    ].notna().sum()
    == 0
):
    raise RuntimeError(
        "SEC facts were standardised, but none could be "
        "linked to a Block 2 issuer_id."
    )

,metric,value
0,standardised_fact_rows,387948
1,fact_rows_with_issuer_id,368521
2,fact_rows_missing_issuer_id,19427
3,fact_issuer_count,84
4,fact_primary_security_count,84
5,filing_rows,9898
6,filing_rows_with_issuer_id,9795
7,filing_rows_missing_issuer_id,103
8,filing_issuer_count,84
9,confirmed_cik_issuer_bridges,84


In [ ]:
# 12. POINT-IN-TIME ACCESS FUNCTIONS
# ------------------------------------------------

def sec_facts_available_as_of(
    facts: pd.DataFrame,
    as_of_date: str | pd.Timestamp,
    selected_only: bool = True,
) -> pd.DataFrame:
    if facts.empty:
        return facts.copy()

    cutoff = pd.Timestamp(as_of_date)
    if cutoff.tzinfo is None:
        cutoff = cutoff.tz_localize("UTC")
    else:
        cutoff = cutoff.tz_convert("UTC")

    out = facts.loc[pd.to_datetime(facts["available_datetime"], utc=True) <= cutoff].copy()
    if selected_only and "is_selected_standard_fact" in out.columns:
        out = out.loc[out["is_selected_standard_fact"]].copy()
    return out


def latest_sec_facts_as_of(
    facts: pd.DataFrame,
    as_of_date: str | pd.Timestamp,
) -> pd.DataFrame:
    available = sec_facts_available_as_of(facts, as_of_date, selected_only=True)
    if available.empty:
        return available

    group_cols = [c for c in ["security_id", "issuer_id", "cik", "standard_concept", "period_end", "unit"] if c in available.columns]
    return (
        available
        .sort_values("available_datetime")
        .groupby(group_cols, dropna=False, as_index=False)
        .tail(1)
        .reset_index(drop=True)
    )

# Example after execution:
# fundamentals_at_2022_12_31_df = latest_sec_facts_as_of(
#     sec_fundamentals_security_linked_df,
#     "2022-12-31",
# )

In [ ]:
# 13. QUALITY CONTROL, COVERAGE AND AVAILABILITY REPORTS
# ------------------------------------------------

def build_sec_coverage_report(
    universe: pd.DataFrame,
    filings: pd.DataFrame,
    facts: pd.DataFrame,
) -> pd.DataFrame:

    base = universe[
        [
            column
            for column in [
                "security_id",
                "issuer_id",
                "ticker",
                "resolved_cik",
                "sec_company_name",
            ]
            if column in universe.columns
        ]
    ].drop_duplicates()

    if filings.empty:
        filing_summary = pd.DataFrame(
            columns=[
                "cik",
                "filing_count",
                "first_filing_date",
                "last_filing_date",
            ]
        )
    else:
        filing_summary = (
            filings
            .groupby("cik", as_index=False)
            .agg(
                filing_count=(
                    "accession_number",
                    "nunique",
                ),
                first_filing_date=(
                    "filing_date",
                    "min",
                ),
                last_filing_date=(
                    "filing_date",
                    "max",
                ),
            )
        )

    if facts.empty:
        fact_summary = pd.DataFrame(
            columns=[
                "cik",
                "standard_fact_count",
                "standard_concept_count",
            ]
        )
    else:
        fact_summary = (
            facts
            .groupby("cik", as_index=False)
            .agg(
                standard_fact_count=(
                    "reported_value",
                    "size",
                ),
                standard_concept_count=(
                    "standard_concept",
                    "nunique",
                ),
                first_fact_available=(
                    "available_datetime",
                    "min",
                ),
                last_fact_available=(
                    "available_datetime",
                    "max",
                ),
            )
        )

    output = base.merge(
        filing_summary,
        left_on="resolved_cik",
        right_on="cik",
        how="left",
    )

    output = output.merge(
        fact_summary,
        left_on="resolved_cik",
        right_on="cik",
        how="left",
        suffixes=("", "_facts"),
    )

    output["has_sec_filings"] = (
        output["filing_count"].fillna(0).gt(0)
    )

    output["has_standardised_facts"] = (
        output["standard_fact_count"].fillna(0).gt(0)
    )

    return output


sec_coverage_report_df = build_sec_coverage_report(
    sec_resolved_universe_df,
    sec_filing_metadata_df,
    sec_fundamentals_standardised_df,
)

sec_linkage_coverage_df = (
    sec_fundamentals_standardised_df
    .groupby(
        [
            "issuer_link_status",
        ],
        dropna=False,
    )
    .agg(
        fact_rows=(
            "reported_value",
            "size",
        ),
        cik_count=(
            "cik",
            "nunique",
        ),
        issuer_count=(
            "issuer_id",
            "nunique",
        ),
        security_count=(
            "security_id",
            "nunique",
        ),
        concept_count=(
            "standard_concept",
            "nunique",
        ),
    )
    .reset_index()
)

sec_fact_duplicate_report_df = (
    sec_fundamentals_mapped_df[
        sec_fundamentals_mapped_df[
            "concept_selection_rank"
        ].gt(1)
    ].copy()
    if not sec_fundamentals_mapped_df.empty
    else pd.DataFrame()
)

sec_unit_report_df = (
    sec_fundamentals_standardised_df
    .groupby(
        ["standard_concept", "unit"],
        dropna=False,
    )
    .size()
    .rename("observation_count")
    .reset_index()
    if not sec_fundamentals_standardised_df.empty
    else pd.DataFrame()
)

sec_form_coverage_df = (
    sec_filing_metadata_df
    .groupby(
        "form",
        dropna=False,
    )
    .agg(
        filing_count=(
            "accession_number",
            "nunique",
        ),
        issuer_count=(
            "cik",
            "nunique",
        ),
    )
    .reset_index()
    if not sec_filing_metadata_df.empty
    else pd.DataFrame()
)


# ------------------------------------------------
# STANDARD-CONCEPT COVERAGE
# ------------------------------------------------

sec_standard_concept_coverage_df = (
    sec_fundamentals_standardised_df
    .groupby(
        [
            "standard_concept",
            "statement_type",
            "core_tier",
        ],
        dropna=False,
    )
    .agg(
        fact_rows=("reported_value", "size"),
        issuer_count=("cik", "nunique"),
        filing_count=("accession_number", "nunique"),
        earliest_period=("period_end", "min"),
        latest_period=("period_end", "max"),
        numeric_fact_share=(
            "reported_value",
            lambda series: pd.to_numeric(
                series,
                errors="coerce",
            ).notna().mean(),
        ),
        period_match_share=(
            "period_type_match",
            "mean",
        ),
        unit_match_share=(
            "unit_family_match",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "core_tier",
            "issuer_count",
            "fact_rows",
        ],
        ascending=[
            True,
            False,
            False,
        ],
    )
    if not sec_fundamentals_standardised_df.empty
    else pd.DataFrame()
)


# ------------------------------------------------
# ALL-CANONICAL-CONCEPT AVAILABILITY
# ------------------------------------------------

canonical_summary = (
    sec_standard_concept_dictionary_df
    .sort_values(
        ["standard_concept", "priority"]
        if "priority" in sec_standard_concept_dictionary_df.columns
        else ["standard_concept"]
    )
    .drop_duplicates("standard_concept")
    [
        [
            "standard_concept",
            "statement_type",
            "expected_period_type",
            "expected_unit_family",
            "core_tier",
            "is_core",
            "aggregation_policy",
        ]
    ]
    .reset_index(drop=True)
)

if sec_fundamentals_security_linked_df.empty:

    observed_availability = pd.DataFrame(
        columns=["standard_concept"]
    )

else:

    observed_availability = (
        sec_fundamentals_security_linked_df
        .groupby(
            "standard_concept",
            dropna=False,
        )
        .agg(
            issuer_coverage=(
                "issuer_id",
                lambda series: series.dropna().nunique(),
            ),
            security_coverage=(
                "security_id",
                lambda series: series.dropna().nunique(),
            ),
            filing_coverage=(
                "accession_number",
                lambda series: series.dropna().nunique(),
            ),
            fact_rows=(
                "reported_value",
                "size",
            ),
            first_reporting_date=(
                "period_end",
                "min",
            ),
            last_reporting_date=(
                "period_end",
                "max",
            ),
            first_available_datetime=(
                "available_datetime",
                "min",
            ),
            last_available_datetime=(
                "available_datetime",
                "max",
            ),
            numeric_fact_share=(
                "reported_value",
                lambda series: pd.to_numeric(
                    series,
                    errors="coerce",
                ).notna().mean(),
            ),
        )
        .reset_index()
    )


sec_standard_concept_availability_df = (
    canonical_summary
    .merge(
        observed_availability,
        on="standard_concept",
        how="left",
    )
)

for count_column in [
    "issuer_coverage",
    "security_coverage",
    "filing_coverage",
    "fact_rows",
]:
    sec_standard_concept_availability_df[count_column] = (
        sec_standard_concept_availability_df[
            count_column
        ]
        .fillna(0)
        .astype(int)
    )

total_sec_issuers = max(
    int(
        sec_resolved_universe_df["issuer_id"]
        .dropna()
        .nunique()
    ),
    1,
)

sec_standard_concept_availability_df[
    "issuer_coverage_rate"
] = (
    sec_standard_concept_availability_df[
        "issuer_coverage"
    ]
    / total_sec_issuers
)

sec_standard_concept_availability_df[
    "observed_in_current_universe"
] = (
    sec_standard_concept_availability_df[
        "fact_rows"
    ] > 0
)

sec_standard_concept_availability_df[
    "coverage_class"
] = pd.cut(
    sec_standard_concept_availability_df[
        "issuer_coverage_rate"
    ],
    bins=[
        -0.001,
        0.10,
        0.30,
        0.60,
        0.80,
        1.00,
    ],
    labels=[
        "VERY_SPARSE",
        "SPARSE",
        "MODERATE",
        "HIGH",
        "VERY_HIGH",
    ],
)

sec_standard_concept_availability_df[
    "first_reporting_year"
] = pd.to_datetime(
    sec_standard_concept_availability_df[
        "first_reporting_date"
    ],
    errors="coerce",
).dt.year.astype("Int64")

sec_standard_concept_availability_df[
    "last_reporting_year"
] = pd.to_datetime(
    sec_standard_concept_availability_df[
        "last_reporting_date"
    ],
    errors="coerce",
).dt.year.astype("Int64")

sec_standard_concept_availability_df = (
    sec_standard_concept_availability_df
    .sort_values(
        [
            "core_tier",
            "issuer_coverage_rate",
            "filing_coverage",
            "standard_concept",
        ],
        ascending=[
            True,
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


sec_mapping_quality_df = pd.DataFrame({
    "metric": [
        "canonical_standard_concepts",
        "source_mapping_rows",
        "mapped_candidate_fact_rows",
        "selected_standardised_fact_rows",
        "unique_standard_concepts_observed",
        "period_type_match_share",
        "unit_family_match_share",
        "unmapped_concepts_in_inventory",
        "automotive_extension_candidates",
    ],
    "value": [
        sec_standard_concept_dictionary_df[
            "standard_concept"
        ].nunique(),
        len(sec_standard_concept_dictionary_df),
        len(sec_fundamentals_mapped_df),
        len(sec_fundamentals_standardised_df),
        (
            sec_fundamentals_standardised_df[
                "standard_concept"
            ].nunique()
            if not sec_fundamentals_standardised_df.empty
            else 0
        ),
        (
            sec_fundamentals_standardised_df[
                "period_type_match"
            ].mean()
            if not sec_fundamentals_standardised_df.empty
            else np.nan
        ),
        (
            sec_fundamentals_standardised_df[
                "unit_family_match"
            ].mean()
            if not sec_fundamentals_standardised_df.empty
            else np.nan
        ),
        len(sec_unmapped_concept_inventory_df),
        len(sec_automotive_extension_candidates_df),
    ],
})

display(sec_mapping_quality_df)
display(sec_standard_concept_availability_df.head(100))


,metric,value
0,canonical_standard_concepts,100.000000
1,source_mapping_rows,147.000000
2,mapped_candidate_fact_rows,402604.000000
3,selected_standardised_fact_rows,387948.000000
4,unique_standard_concepts_observed,84.000000
5,period_type_match_share,0.999876
6,unit_family_match_share,0.998345
7,unmapped_concepts_in_inventory,7315.000000
8,automotive_extension_candidates,175.000000


,standard_concept,statement_type,expected_period_type,expected_unit_family,core_tier,is_core,aggregation_policy,issuer_coverage,security_coverage,filing_coverage,fact_rows,first_reporting_date,last_reporting_date,first_available_datetime,last_available_datetime,numeric_fact_share,issuer_coverage_rate,observed_in_current_universe,coverage_class,first_reporting_year,last_reporting_year
0,total_assets,BALANCE_SHEET,INSTANT,MONETARY,1,True,LATEST_INSTANT,84,84,3507,7643,2008-06-29,2026-06-30,2009-07-22 00:00:00+00:00,2026-07-17 13:35:42+00:00,1.0,0.933333,True,VERY_HIGH,2008,2026
1,total_equity,BALANCE_SHEET,INSTANT,MONETARY,1,True,LATEST_INSTANT,84,84,3507,14299,2006-09-24,2026-06-30,2009-07-22 00:00:00+00:00,2026-07-17 13:35:42+00:00,1.0,0.933333,True,VERY_HIGH,2006,2026
2,net_income,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,84,84,3462,15213,2007-09-28,2026-06-30,2009-07-22 00:00:00+00:00,2026-07-17 13:35:42+00:00,1.0,0.933333,True,VERY_HIGH,2007,2026
3,income_tax_expense,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,84,84,3259,11021,2007-09-28,2026-06-30,2009-07-22 00:00:00+00:00,2026-07-17 13:35:42+00:00,1.0,0.933333,True,VERY_HIGH,2007,2026
4,profit_before_tax,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,84,84,3181,11028,2007-09-28,2026-06-30,2009-07-22 00:00:00+00:00,2026-07-17 13:35:42+00:00,1.0,0.933333,True,VERY_HIGH,2007,2026
5,basic_eps,INCOME_STATEMENT,DURATION,PER_SHARE,1,True,PERIOD_VALUE,84,84,3086,12219,2007-09-28,2026-06-30,2009-07-22 00:00:00+00:00,2026-07-17 13:35:42+00:00,1.0,0.933333,True,VERY_HIGH,2007,2026
6,diluted_eps,INCOME_STATEMENT,DURATION,PER_SHARE,1,True,PERIOD_VALUE,84,84,3053,12289,2007-09-28,2026-06-30,2009-07-22 00:00:00+00:00,2026-07-17 13:35:42+00:00,1.0,0.933333,True,VERY_HIGH,2007,2026
7,cash_and_cash_equivalents,BALANCE_SHEET,INSTANT,MONETARY,1,True,LATEST_INSTANT,83,83,3397,13528,2006-09-24,2026-06-30,2009-07-22 00:00:00+00:00,2026-07-17 13:35:42+00:00,1.0,0.922222,True,VERY_HIGH,2006,2026
8,operating_cash_flow,CASH_FLOW,DURATION,MONETARY,1,True,PERIOD_VALUE,83,83,2962,7007,2007-09-28,2026-06-30,2009-07-22 00:00:00+00:00,2026-07-17 13:35:42+00:00,1.0,0.922222,True,VERY_HIGH,2007,2026
9,revenue,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,82,82,3023,11980,2007-09-28,2026-06-30,2009-07-22 00:00:00+00:00,2026-07-17 13:35:42+00:00,1.0,0.911111,True,VERY_HIGH,2007,2026


In [ ]:
# 14. BLOCK 3 OUTPUT CONTRACT
# ------------------------------------------------

block_3_data = {
    # SEC directory and bridges
    "sec_ticker_directory_df": sec_ticker_directory_df,
    "security_master_normalised_df": security_master_normalised_df,
    "identifier_candidates_df": identifier_candidates_df,
    "sec_universe_seed_df": sec_universe_seed_df,
    "sec_universe_map_df": sec_universe_map_df,
    "sec_resolved_universe_df": sec_resolved_universe_df,
    "sec_unresolved_universe_df": sec_unresolved_universe_df,
    "sec_cik_security_bridge_df": sec_cik_security_bridge_df,
    "sec_cik_issuer_bridge_candidates_df": sec_cik_issuer_bridge_candidates_df,
    "sec_cik_issuer_bridge_df": sec_cik_issuer_bridge_df,
    "sec_cik_issuer_conflicts_df": sec_cik_issuer_conflicts_df,

    # Issuer-centric downstream contracts
    "usa_economic_issuer_universe_df": usa_economic_issuer_universe_df,
    "usa_issuer_security_universe_df": usa_issuer_security_universe_df,
    "usa_entity_relationship_graph_df": usa_entity_relationship_graph_df,
    "usa_preferred_accounting_source_df": usa_preferred_accounting_source_df,

    # Raw SEC data
    "sec_submissions_raw_df": sec_submissions_raw_df,
    "sec_companyfacts_raw_df": sec_companyfacts_raw_df,
    "sec_facts_pit_df": sec_facts_pit_df,

    # Filing outputs
    "sec_filing_metadata_cik_df": sec_filing_metadata_cik_df,
    "sec_filing_metadata_df": sec_filing_metadata_df,
    "sec_filing_metadata_linked_df": sec_filing_metadata_linked_df,
    "usa_filing_metadata_df": usa_filing_metadata_df,

    # Canonical mapping
    "sec_standard_concept_dictionary_df": sec_standard_concept_dictionary_df,
    "usa_standard_concept_dictionary_df": usa_standard_concept_dictionary_df,
    "sec_fundamentals_mapped_df": sec_fundamentals_mapped_df,
    "sec_fundamentals_standardised_cik_df": sec_fundamentals_standardised_cik_df,
    "sec_fundamentals_standardised_df": sec_fundamentals_standardised_df,
    "sec_fundamentals_security_linked_df": sec_fundamentals_security_linked_df,
    "usa_fundamentals_standardised_df": usa_fundamentals_standardised_df,

    # Mapping and coverage QA
    "sec_unmapped_concept_inventory_df": sec_unmapped_concept_inventory_df,
    "sec_automotive_extension_candidates_df": sec_automotive_extension_candidates_df,
    "sec_standard_concept_coverage_df": sec_standard_concept_coverage_df,
    "sec_standard_concept_availability_df": sec_standard_concept_availability_df,
    "sec_mapping_quality_df": sec_mapping_quality_df,
    "sec_coverage_report_df": sec_coverage_report_df,
    "sec_linkage_coverage_df": sec_linkage_coverage_df,
    "sec_issuer_link_quality_df": sec_issuer_link_quality_df,
    "sec_fact_duplicate_report_df": sec_fact_duplicate_report_df,
    "sec_unit_report_df": sec_unit_report_df,
    "sec_form_coverage_df": sec_form_coverage_df,

    # Download logs
    "sec_submission_download_log_df": sec_submission_download_log_df,
    "sec_companyfacts_download_log_df": sec_companyfacts_download_log_df,
}

print(
    "Block 3 transformations complete."
)

print(
    "\nPrincipal outputs:"
)

for name in [
    "sec_cik_issuer_bridge_df",
    "sec_filing_metadata_df",
    "sec_facts_pit_df",
    "sec_fundamentals_standardised_df",
    "usa_fundamentals_standardised_df",
    "sec_coverage_report_df",
]:
    dataframe = block_3_data[
        name
    ]

    print(
        f"  {name}: "
        f"{len(dataframe):,} rows"
    )

display(
    sec_issuer_link_quality_df
)

display(
    sec_coverage_report_df.head(20)
)

Block 3 transformations complete.

Principal outputs:
  sec_cik_issuer_bridge_df: 84 rows
  sec_filing_metadata_df: 9,898 rows
  sec_facts_pit_df: 1,306,194 rows
  sec_fundamentals_standardised_df: 387,948 rows
  usa_fundamentals_standardised_df: 387,948 rows
  sec_coverage_report_df: 93 rows


,metric,value
0,standardised_fact_rows,387948
1,fact_rows_with_issuer_id,368521
2,fact_rows_missing_issuer_id,19427
3,fact_issuer_count,84
4,fact_primary_security_count,84
5,filing_rows,9898
6,filing_rows_with_issuer_id,9795
7,filing_rows_missing_issuer_id,103
8,filing_issuer_count,84
9,confirmed_cik_issuer_bridges,84


,security_id,issuer_id,ticker,resolved_cik,sec_company_name,cik,filing_count,first_filing_date,last_filing_date,cik_facts,standard_fact_count,standard_concept_count,first_fact_available,last_fact_available,has_sec_filings,has_standardised_facts
0,GAS_021CBE637C40EA92408A,GAI_2F165C93319FA20296BA,MXL,0001288469,"MAXLINEAR, INC",0001288469,38,2017-05-09,2026-04-23,0001288469,7016,55,2011-07-28 00:00:00+00:00,2026-04-23 20:13:32+00:00,True,True
1,GAS_0222D68E994B13D9EF6C,GAI_766E3386DD1DD1EF6C87,CRUS,0000772406,"CIRRUS LOGIC, INC.",0000772406,54,2013-05-29,2026-05-21,0000772406,6460,53,2011-07-25 00:00:00+00:00,2026-05-21 20:11:56+00:00,True,True
2,GAS_037293E03ADDC845CF66,GAI_D9C9C458A4EB058F3B1C,OUST,0001816581,"Ouster, Inc.",0001816581,26,2020-11-13,2026-05-05,0001816581,2652,54,2020-11-13 22:22:06+00:00,2026-05-05 21:08:54+00:00,True,True
3,GAS_06A91B1963440456046E,GAI_B0B2DA0C6FF143ACE220,CON,0002014596,"Concentra Group Holdings Parent, Inc.",0002014596,8,2024-08-27,2026-05-07,0002014596,847,46,2024-08-27 20:32:12+00:00,2026-05-07 21:03:26+00:00,True,True
4,GAS_06C0D65953984F4526CB,GAI_3A97C44308BBACE2EA3C,BIDU,0001329099,"Baidu, Inc.",0001329099,281,2005-08-24,2026-07-16,0001329099,3297,61,2010-03-26 11:52:41+00:00,2026-03-17 10:30:12+00:00,True,True
5,GAS_097275C11AE3D771B6BE,GAI_E5EB845467862C2BD5B3,INDI,0001841925,"indie Semiconductor, Inc.",0001841925,21,2021-08-13,2026-05-11,0001841925,2413,52,2021-08-13 20:11:24+00:00,2026-05-08 21:35:47+00:00,True,True
6,GAS_1050039D64860158AA83,GAI_5586FB58001D48BE17BA,XPEV,0001810997,XPENG INC.,0001810997,200,2020-09-28,2026-07-15,0001810997,863,54,2021-04-16 12:00:22+00:00,2026-04-16 10:09:13+00:00,True,True
7,GAS_10E1F7E1A92F5395CA8C,GAI_9C6A4F69CFB8EEC56A28,MU,0000723125,MICRON TECHNOLOGY INC,0000723125,37,2017-06-30,2026-06-25,0000723125,7980,62,2010-01-12 00:00:00+00:00,2026-06-24 22:59:46+00:00,True,True
8,GAS_1337773DABE8069B0813,GAI_44EC82C746E0F14F6FB3,AMPX,0001899287,"Amprius Technologies, Inc.",0001899287,17,2022-05-17,2026-05-07,0001899287,1561,44,2022-05-16 21:48:31+00:00,2026-05-07 20:52:27+00:00,True,True
9,GAS_19A0146A015807F45BF4,GAI_044EBD9F4107A05EB8ED,NIU,0001744781,Niu Technologies,0001744781,76,2018-11-20,2026-07-06,0001744781,1463,50,2019-04-25 10:10:36+00:00,2026-04-17 10:12:40+00:00,True,True


In [ ]:
# 15. PERSIST BLOCK 3 OUTPUTS
# ------------------------------------------------

def make_parquet_safe(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Return a Parquet-safe copy while preserving datetime and numeric columns.
    """
    output = dataframe.copy()

    for column in output.columns:
        if output[column].dtype == "object":
            non_missing = output[column].dropna()

            if (
                not non_missing.empty
                and non_missing.map(type).nunique() > 1
            ):
                output[column] = output[column].astype("string")

    return output


def persist_dataframe(
    name: str,
    dataframe: pd.DataFrame,
    output_dir: Path,
    *,
    overwrite: bool = True,
) -> dict:
    """Persist one DataFrame as compressed Parquet."""

    output_path = output_dir / f"{name}.parquet"

    if output_path.exists() and not overwrite:
        raise FileExistsError(
            f"Refusing to overwrite existing output: {output_path}"
        )

    safe_df = make_parquet_safe(dataframe)

    safe_df.to_parquet(
        output_path,
        index=False,
        engine="pyarrow",
        compression="snappy",
    )

    return {
        "table_name": name,
        "path": str(output_path),
        "row_count": int(len(safe_df)),
        "column_count": int(len(safe_df.columns)),
        "columns": list(map(str, safe_df.columns)),
        "file_size_bytes": int(output_path.stat().st_size),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }


def load_block_3_outputs(
    output_dir: Path = BLOCK_3_OUTPUT_DIR,
) -> dict[str, pd.DataFrame]:
    """Load persisted Block 3 tables from the manifest."""

    manifest_path = output_dir / "block_3_manifest.json"

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Block 3 manifest not found: {manifest_path}"
        )

    with manifest_path.open("r", encoding="utf-8") as file:
        manifest = json.load(file)

    loaded = {}

    for table in manifest["tables"]:
        table_path = Path(table["path"])

        if not table_path.exists():
            raise FileNotFoundError(
                f"Manifest table is missing: {table_path}"
            )

        loaded[table["table_name"]] = pd.read_parquet(table_path)

    return loaded


if PERSIST_BLOCK_3_OUTPUTS:

    manifest_rows = []

    for table_name, dataframe in block_3_data.items():
        manifest_rows.append(
            persist_dataframe(
                table_name,
                dataframe,
                BLOCK_3_OUTPUT_DIR,
                overwrite=OVERWRITE_PERSISTED_OUTPUTS,
            )
        )

    block_3_manifest = {
        "block": 3,
        "block_name": "US SEC point-in-time fundamentals with expanded canonical schema",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "project_root": str(PROJECT_ROOT),
        "input_manifest": str(BLOCK_2_MANIFEST_PATH),
        "output_directory": str(BLOCK_3_OUTPUT_DIR),
        "source_system": "SEC EDGAR submissions and Company Facts APIs",
        "allowed_forms": sorted(ALLOWED_FORMS),
        "as_of_date": AS_OF_DATE,
        "tables": manifest_rows,
    }

    with BLOCK_3_MANIFEST_PATH.open("w", encoding="utf-8") as file:
        json.dump(block_3_manifest, file, indent=2)

    block_3_persistence_report_df = pd.DataFrame(manifest_rows)

    print("Block 3 outputs persisted successfully.")
    print("Manifest:", BLOCK_3_MANIFEST_PATH)

    display(
        block_3_persistence_report_df[
            [
                "table_name",
                "row_count",
                "column_count",
                "file_size_bytes",
                "path",
            ]
        ]
    )

else:

    block_3_persistence_report_df = pd.DataFrame()

    print(
        "PERSIST_BLOCK_3_OUTPUTS is False. "
        "Outputs remain available only in the current runtime."
    )


# ------------------------------------------------
# 16. PERSISTENCE VALIDATION
# ------------------------------------------------

if PERSIST_BLOCK_3_OUTPUTS:

    reloaded_block_3_data = load_block_3_outputs()

    required_downstream_tables = {
        "sec_cik_security_bridge_df",
        "sec_cik_issuer_bridge_df",
        "usa_economic_issuer_universe_df",
        "usa_issuer_security_universe_df",
        "usa_preferred_accounting_source_df",
        "sec_filing_metadata_df",
        "usa_filing_metadata_df",
        "sec_fundamentals_standardised_df",
        "usa_fundamentals_standardised_df",
        "sec_standard_concept_dictionary_df",
        "usa_standard_concept_dictionary_df",
        "sec_issuer_link_quality_df",
        "sec_coverage_report_df",
        "sec_standard_concept_coverage_df",
        "sec_standard_concept_availability_df",
        "sec_unmapped_concept_inventory_df",
    }

    missing_downstream_tables = required_downstream_tables.difference(
        reloaded_block_3_data.keys()
    )

    if missing_downstream_tables:
        raise RuntimeError(
            "Persistence validation failed. Missing tables: "
            f"{sorted(missing_downstream_tables)}"
        )

    for required_name in required_downstream_tables:
        original_rows = len(block_3_data[required_name])
        reloaded_rows = len(reloaded_block_3_data[required_name])

        if original_rows != reloaded_rows:
            raise RuntimeError(
                f"Row-count mismatch for {required_name}: "
                f"{original_rows} original versus {reloaded_rows} reloaded."
            )

    print(
        "Persistence validation passed. "
        "Later modules can load Block 3 without rerunning SEC collection."
    )

Block 3 outputs persisted successfully.
Manifest: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_3/block_3_manifest.json


,table_name,row_count,column_count,file_size_bytes,path
0,sec_ticker_directory_df,10426,4,295574,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
1,security_master_normalised_df,512,9,42723,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
2,identifier_candidates_df,22376,9,88130,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
3,sec_universe_seed_df,825,12,61108,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
4,sec_universe_map_df,825,21,70990,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
5,sec_resolved_universe_df,93,21,24201,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
6,sec_unresolved_universe_df,732,21,58444,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
7,sec_cik_security_bridge_df,93,12,17142,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
8,sec_cik_issuer_bridge_candidates_df,93,16,19596,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
9,sec_cik_issuer_bridge_df,84,11,16257,/content/drive/MyDrive/Colab Notebooks/00 A1 A...


Persistence validation passed. Later modules can load Block 3 without rerunning SEC collection.


## Architectural notes

### Authoritative entity bridge

SEC Company Facts are issuer-level observations keyed by CIK. Block 3 therefore
builds a confirmed CIK-to-issuer bridge and attaches `issuer_id` to the canonical
standardised facts before persistence.

### Security treatment

One issuer may have several listed securities. The canonical SEC facts retain the
issuer ID and a primary-security reference. A separate security-expanded table
preserves all mapped securities without multiplying the authoritative issuer-level
fact table.

### Downstream contract

Block 10 should load `usa_fundamentals_standardised_df`,
`usa_filing_metadata_df`, `usa_economic_issuer_universe_df` and
`usa_issuer_security_universe_df`.
